Download all spectra as .fits files and save a 'catalog' .fits file
Does some filtering for redshift, flagging etc

In [1]:
import requests
from bs4 import BeautifulSoup
from astropy.io import fits
import matplotlib.pyplot as plt
from desispec.io.spectra import read_spectra
from desispec.coaddition import coadd_cameras
import numpy as np
from desitarget.targetmask import desi_mask
from astropy.table import Table

In [2]:
r_i = requests.get("https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/") 
soup_i = BeautifulSoup(r_i.content, "html.parser")
all_links_i = soup_i.find_all("a")
linkList_i = [link['href'] for link in all_links_i[1:]] # creates a list of all the first indicies of the coaad files

all_lists = []
for i in range(len(linkList_i)):
    r_j = requests.get("https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/" + str(linkList_i[i]))
    soup_j = BeautifulSoup(r_j.content, "html.parser")
    all_links_j = soup_j.find_all("a")
    linkList_j = [link['href'] for link in all_links_j] 
    all_lists.append(linkList_j[1:]) # creates a list of all the second indicies of the coaad files for each of the first indicies
    

In [3]:
tile = set()
zArray = []
tidArray = []
hdul = fits.open("qso_cat_dr1_main_dark_healpix_zlya-v0.fits") # lya quasar catalog
data = hdul[1].data

In [5]:
for i in range(len(linkList_i)):
    for j in range(len(all_lists[i])):
        if int(all_lists[i][j].replace("/","")) in tile: # checks if tile has already been downloaded and all spectra extracted
            print("we in this")
            continue
        else:
            downFile = ('https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/' +  str(linkList_i[i]) # uses url generated from linkList and all_lists
                        + str(all_lists[i][j]) + 'coadd-main-dark-'+  str(all_lists[i][j]).replace("/","") + '.fits')
            !wget "{downFile}"
            print(all_lists[i][j].replace("/",""))

            spec = read_spectra("coadd-main-dark-" + all_lists[i][j].replace("/","") + ".fits" ) # gets spectra data from the coaad file
            fibermap = spec.fibermap
            coadd_dict = {tid: ii for ii, tid in enumerate(fibermap["TARGETID"])} # creates a dictionary of indicies with the target id as the key

            for ii, tid in enumerate(data["TARGETID"]):
                if (data["Z"][ii] <5.4) and (data["ZWARN"][ii] == 0) and (data["BI_CIV"][ii] == 0): #checks that Z > Z_min, ZWARN = 0, and BI_CIV > BI_CIV_min
                    if tid in coadd_dict: # checks that the target id from the lya QSO catelog is in the coadd file
                        tidArray.append(tid) #generates a list of target id's of valis QSOs
                        zArray.append(data["Z"][ii]) # generates a list of redshift of all the valid QSOs
                        spec_one = spec[coadd_dict[tid]]
                        wave = coadd_cameras(spec_one).wave['brz'] # combines the the three arms
                        flux = coadd_cameras(spec_one).flux['brz'][0]
                        ivar = coadd_cameras(spec_one).ivar['brz'][0]
                        t = Table()
                        t['WAVE'] = wave # formats the data into a table
                        t['FLUX'] = flux
                        t['IVAR'] = ivar
                        t.write(str(tid) + '.fits',"overwrite=True") # writes the data to a fits file for analysis

            tile.add(int(all_lists[i][j].replace("/",""))) # used to make sure that if code stops while downloading, the last downloaded tile is saved so we dont have to restart
            filename = "coadd-main-dark-" + all_lists[i][j].replace("/","") + ".fits"
            !rm -rf "{filename}" # removes the downloaded coadd file
        
tZ = Table()
tZ['TARGETID'] = tidArray 
tZ['Z'] = zArray
tZ.write('valid_QSO_Z.fits',"overwrite=True") # writes a file with tid and redshifts of all the valid QSOs


we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this
we in this

In [6]:
print(tile)
print(len(tidArray))


{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 24, 25, 26, 27, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 56, 57, 58, 59, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 146, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 200, 202, 203, 224, 225, 226, 227, 230, 232, 233, 234, 235, 236, 238, 239, 250, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 600, 601, 602, 603, 604, 60